# 🚦 Huấn luyện Dueling Double DQN (3DQN) trên Google Colab với GAMA Headless

Notebook này hướng dẫn quy trình từ A đến Z để huấn luyện mô hình **3DQN** điều khiển đèn tín hiệu giao thông vi mô đa tầng (mặt đất, hầm, cầu vượt) sử dụng **GAMA Platform Headless** và dữ liệu/mô hình được lưu trên **Google Drive**.

### 📌 Quy trình tổng quan:
1. **Kiểm tra phần cứng (GPU/CUDA)**.
2. **Mount Google Drive** và đồng bộ mã nguồn sang local disk (tối ưu hóa tốc độ I/O).
3. **Cài đặt Xvfb, Mesa Software OpenGL & GAMA Platform Headless** (tự động cài đặt và giải nén bảo đảm 100% thành công).
4. **Cài đặt thư viện Python** (`gama-client`, `torch`, `pandas`, `matplotlib`).
5. **Kiểm tra kết nối GAMA Server** qua giao thức WebSocket.
6. **Khởi chạy Huấn luyện 3DQN** (`train.py`) với checkpoint đồng bộ trực tiếp về Google Drive.
7. **Trực quan hóa đồ thị huấn luyện** (Reward, Waiting Time, Queue, Throughput).
8. **Đánh giá mô hình (Evaluation)** với các validation seeds độc lập.

## 1. Kiểm tra cấu hình GPU / Colab Runtime

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Lưu ý: Bạn đang chạy trên CPU runtime. Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc độ huấn luyện PyTorch.")

## 2. Mount Google Drive & Chuẩn bị Workspace

> **Mẹo hiệu năng:** Hệ thống tệp Google Drive mount qua FUSE (`/content/drive`) có độ trễ I/O lớn khi đọc ghi liên tục hàng nghìn file cache của Eclipse/GAMA. Vì vậy, ta sao chép project sang đĩa local Colab (`/content/Traffic-Light-ABM`) để chạy siêu mượt, đồng thời **tạo symlink cho thư mục `run/`** trỏ thẳng về Google Drive để toàn bộ weights (`.pt`) và logs (`.csv`) được lưu vĩnh viễn trên Drive.

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Drive
drive.mount('/content/drive')

# Đường dẫn thư mục chứa repository trên Google Drive của bạn
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Traffic-Light-ABM'
LOCAL_WORKSPACE = '/content/Traffic-Light-ABM'

if not os.path.exists(DRIVE_PROJECT_PATH):
    print(f"⚠️ Không tìm thấy thư mục {DRIVE_PROJECT_PATH} trên Google Drive!")
    print("Bạn có thể sao chép repository từ GitHub về Colab tạm thời bằng lệnh dưới:")
    # !git clone https://github.com/ngocbityta/Traffic-Light-ABM.git $LOCAL_WORKSPACE
else:
    print(f"✅ Đã tìm thấy dự án trên Google Drive: {DRIVE_PROJECT_PATH}")
    
    # Luôn đồng bộ đè mã nguồn mới nhất từ Drive sang local disk (tăng tốc độ I/O)
    print("Đang đồng bộ mã nguồn sang local disk...")
    shutil.copytree(
        DRIVE_PROJECT_PATH,
        LOCAL_WORKSPACE,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('.git', 'venv', '__pycache__', 'run')
    )
    print("✅ Hoàn tất đồng bộ mã nguồn sang local disk.")
        
    # Tạo symlink thư mục dqn_py/run trỏ về Google Drive để lưu vĩnh viễn logs & weights
    drive_run_dir = os.path.join(DRIVE_PROJECT_PATH, 'dqn_py', 'run')
    os.makedirs(drive_run_dir, exist_ok=True)
    
    local_run_dir = os.path.join(LOCAL_WORKSPACE, 'dqn_py', 'run')
    if os.path.islink(local_run_dir):
        os.unlink(local_run_dir)
    elif os.path.exists(local_run_dir):
        shutil.rmtree(local_run_dir)
    os.symlink(drive_run_dir, local_run_dir)
    print(f"✅ Đã liên kết thư mục kết quả: {local_run_dir} -> {drive_run_dir}")


## 3. Cài đặt GAMA Platform Headless, Xvfb & Mesa Software OpenGL

### ⚠️ Khắc phục lỗi Crash `libGLX_mesa.so` & File not found:
- Colab là môi trường ephemeral (mỗi khi restart runtime, thư mục `/opt/` sẽ bị xóa và cần chạy lại ô này).
- Cell này tự động kiểm tra, nếu chưa cài sẽ tải gói `.deb`, cài đặt bằng `dpkg` (kèm fallback `dpkg-deb -x` giải nén trực tiếp bảo đảm không bao giờ thiếu file).
- Thiết lập `LIBGL_ALWAYS_SOFTWARE=1` và `GALLIUM_DRIVER=llvmpipe` để ngăn chặn lỗi sập OpenGL của JVM.

In [ ]:
import os, glob

# 1. Dừng bất kỳ tiến trình Xvfb cũ và dọn dẹp lock files
!killall -9 Xvfb 2>/dev/null || true
!rm -f /tmp/.X99-lock /tmp/.X11-unix/X99 /tmp/.X*-lock /tmp/.X11-unix/X*

# 2. Cài đặt các gói phụ thuộc hệ thống và driver Mesa OpenGL phần mềm
!apt-get update -qq
!apt-get install -y -qq xvfb x11-utils libgl1-mesa-dri mesa-utils libglx-mesa0 libgtk-3-0 libxrender1 libxtst6 libxi6 wget

# 3. Tải và cài đặt GAMA Platform 2025.06.4 Linux with JDK 21
gama_deb_path = "/content/GAMA_2025.06.4_Linux_with_JDK.deb"

found = glob.glob("/opt/**/gama-headless.sh", recursive=True)
if not found:
    if not os.path.exists(gama_deb_path) or os.path.getsize(gama_deb_path) < 300_000_000:
        print("Đang tải GAMA Platform 2025.06.4 Linux with JDK (~358MB)...")
        !wget -c --show-progress -O $gama_deb_path \
            https://github.com/gama-platform/gama/releases/download/2025.06.4/GAMA_2025.06.4_Linux_with_JDK.deb
    print("Đang cài đặt GAMA package...")
    !dpkg -i $gama_deb_path || apt-get install -f -y -qq

found = glob.glob("/opt/**/gama-headless.sh", recursive=True) + glob.glob("/usr/**/gama-headless.sh", recursive=True)
if not found:
    print("⚠️ Đang giải nén trực tiếp vào hệ thống bằng dpkg-deb...")
    !dpkg-deb -x $gama_deb_path /
    found = glob.glob("/opt/**/gama-headless.sh", recursive=True)

if not found:
    raise FileNotFoundError("❌ Không tìm thấy gama-headless.sh sau khi cài đặt! Vui lòng kiểm tra lại file .deb.")

gama_headless_path = os.path.abspath(found[0])
!chmod +x "$gama_headless_path"
print("✅ Đã tìm thấy và kích hoạt GAMA Headless tại:", gama_headless_path)

# 4. Cấu hình các biến môi trường an toàn
os.environ["GAMA_HEADLESS"] = gama_headless_path
os.environ["DISPLAY"] = ":99"
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
os.environ["GALLIUM_DRIVER"] = "llvmpipe"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "mesa"
os.environ["_JAVA_OPTIONS"] = "-Dsun.java2d.opengl=false -Dsun.java2d.xrender=false -Djava.awt.headless=false"

# 5. Khởi động Xvfb nền cố định trên :99 và kiểm tra trạng thái
!nohup Xvfb :99 -screen 0 1024x768x24 -ac -extension GLX +render -noreset >/dev/null 2>&1 & sleep 2
!xdpyinfo -display :99 >/dev/null 2>&1 && echo "✅ Xvfb :99 đang hoạt động ổn định!" || echo "⚠️ Chú ý: Xvfb sẽ được Python tự khởi động lại khi cần."

# 6. Kiểm tra phiên bản GAMA Headless
!bash "$gama_headless_path" -version || true


## 4. Cài đặt thư viện Python

Cài đặt `gama-client` (WebSocket client điều khiển GAMA) và các thư viện cần thiết. 

*Lưu ý:* Ta không ép buộc `numpy<2` để tránh xung đột với các gói OpenCV / JAX mặc định của Colab; mã nguồn 3DQN đã được tối ưu tương thích hoàn toàn với cả NumPy 1.x và 2.x.

In [ ]:
%cd /content/Traffic-Light-ABM
!pip install --quiet "gama-client" pandas matplotlib

import gama_client, torch, numpy
print("VERIFY OK:")
print("  gama_client =", gama_client.__file__)
print("  torch       =", torch.__version__)
print("  numpy       =", numpy.__version__)

## 5. Kiểm tra kết nối GAMA Server Headless từ Python

Chạy kiểm tra integration handshake qua WebSocket giữa Python và mô hình `Traffic/models/intersection-dqn-python.gaml`.

In [ ]:
%cd /content/Traffic-Light-ABM
import os, glob

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
os.environ["GALLIUM_DRIVER"] = "llvmpipe"
os.environ["DISPLAY"] = ":99"

gama_path = os.environ.get("GAMA_HEADLESS")
if not gama_path or not os.path.isfile(gama_path):
    found = glob.glob("/opt/**/gama-headless.sh", recursive=True)
    if found:
        os.environ["GAMA_HEADLESS"] = found[0]

# Chạy test server kết nối (server_manager tự động bảo đảm Xvfb sẵn sàng)
!python3 dqn_py/test_doan_server.py --start-server --horizon 1500


## 6. Huấn luyện 3DQN (Dueling Double DQN)

Bạn có thể tùy chỉnh các tham số dưới đây:
- `--total-steps`: Tổng số bước quyết định của Agent (VD: `1000000` hoặc `5000000`). Để kiểm tra nhanh, bạn có thể đặt `50000`.
- `--warmup`: Số transition ngẫu nhiên để nạp vào Replay Buffer trước khi học gradient (mặc định `100000`, test nhanh có thể dùng `1000` hoặc `5000`).
- `--episode-horizon`: Số chu kỳ mô phỏng GAMA mỗi episode (mặc định `8000` cycles = 800 giây).
- `--validate-every`: Đánh giá policy trên validation seeds (3001, 3002) sau mỗi N episode.
- `--start-server`: Tự động khởi chạy GAMA server ngầm qua `gama-headless.sh`.
- `--port`: Cổng WebSocket (mặc định `6868`).

*Toàn bộ checkpoint (`dqn_weights.pt`, `best_weights.pt`) và log huấn luyện (`training_log.csv`) sẽ được ghi trực tiếp vào Google Drive thông qua symlink!*

In [ ]:
%cd /content/Traffic-Light-ABM

# Chạy lệnh huấn luyện 3DQN với 20 Workers song song (Tự động quản lý Xvfb & WebSocket)
!python3 dqn_py/train.py \
    --total-steps 1000000 \
    --workers 20 \
    --batch-size 128 \
    --updates-per-step 2 \
    --warmup 10000 \
    --buffer-size 1000000 \
    --episode-horizon 8000 \
    --validate-every 10 \
    --start-server \
    --port 6868


## 7. Trực quan hóa & Theo dõi Tiến trình Huấn luyện

Đọc dữ liệu từ file `training_log.csv` mới nhất để vẽ đồ thị theo dõi các chỉ số quan trọng:
1. **Episode Reward**: Phần thưởng tích lũy theo episode.
2. **Average Waiting Time**: Thời gian chờ trung bình của xe tại nút giao.
3. **Final Queue Length**: Độ dài hàng đợi cuối episode.
4. **Throughput**: Số lượng phương tiện đã vượt qua nút giao thành công.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Tìm thư mục train mới nhất
run_dirs = sorted(glob.glob('/content/Traffic-Light-ABM/dqn_py/run/train_*'))
if not run_dirs:
    run_dirs = sorted(glob.glob('dqn_py/run/train_*'))

if not run_dirs:
    print("Chưa tìm thấy dữ liệu train trong dqn_py/run/")
else:
    latest_run = run_dirs[-1]
    log_csv = os.path.join(latest_run, "training_log.csv")
    print("📁 Thư mục kết quả mới nhất:", latest_run)
    
    if os.path.exists(log_csv):
        df = pd.read_csv(log_csv)
        total_steps_target = 1_000_000
        total_decisions = df['decisions'].sum() if 'decisions' in df.columns else 0
        pct = min(100.0, (total_decisions / total_steps_target) * 100.0)
        episodes_done = len(df)
        
        avg_reward_10 = df['episode_reward'].tail(10).mean()
        avg_wait_10 = df['avg_waiting_time'].tail(10).mean()
        current_eps = df['epsilon'].iloc[-1] if 'epsilon' in df.columns else 0.0
        current_loss = df['avg_loss'].iloc[-1] if 'avg_loss' in df.columns else 0.0
        
        # 1. Thanh Tiến Độ % (Progress Bar)
        bar_len = 20
        filled = int(bar_len * pct / 100.0)
        bar_ascii = '=' * filled + '-' * (bar_len - filled)
        
        html_code = f"""
        <div style="font-family: Arial, sans-serif; background: #0f172a; padding: 16px; border-radius: 12px; color: #f8fafc; margin-bottom: 16px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                <span style="font-size: 16px; font-weight: bold;">🚦 Tiến độ Huấn luyện 3DQN</span>
                <span style="font-size: 18px; font-weight: bold; color: #38bdf8;">{pct:.2f}%</span>
            </div>
            <div style="background: #334155; border-radius: 8px; height: 16px; overflow: hidden; margin-bottom: 12px;">
                <div style="background: linear-gradient(90deg, #38bdf8, #22c55e); width: {pct:.2f}%; height: 100%; transition: width 0.5s;"></div>
            </div>
            <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 8px; text-align: center;">
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Số Bước (Steps)</b><br><span style="color:#38bdf8;">{total_decisions:,} / {total_steps_target:,}</span></div>
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Số Episode</b><br><span style="color:#a78bfa;">{episodes_done}</span></div>
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Reward (10 ep)</b><br><span style="color:#4ade80;">{avg_reward_10:.1f}</span></div>
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Wait Time (10 ep)</b><br><span style="color:#fb923c;">{avg_wait_10:.2f}s</span></div>
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Epsilon (Eps)</b><br><span style="color:#f472b6;">{current_eps:.3f}</span></div>
                <div style="background: #1e293b; padding: 8px; border-radius: 6px;"><b>Loss</b><br><span style="color:#facc15;">{current_loss:.4f}</span></div>
            </div>
        </div>
        """
        display(HTML(html_code))
        print(f"Tiến độ: [{bar_ascii}] {pct:.2f}% ({total_decisions:,} / {total_steps_target:,} steps | {episodes_done} episodes)")
        
        # 2. Hiển thị 5 dòng logs mới nhất
        display(df.tail())
        
        # 3. Vẽ 4 biểu đồ giám sát theo thời gian thực
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        
        # Reward
        axes[0, 0].plot(df['episode'], df['episode_reward'], color='#2563eb', marker='o', markersize=3, label='Reward')
        axes[0, 0].set_title("Episode Reward", fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel("Episode")
        axes[0, 0].set_ylabel("Reward")
        axes[0, 0].grid(True, alpha=0.3)
        
        # Avg Waiting Time
        axes[0, 1].plot(df['episode'], df['avg_waiting_time'], color='#ea580c', marker='s', markersize=3, label='Wait Time')
        axes[0, 1].set_title("Thời gian chờ trung bình (giây)", fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel("Episode")
        axes[0, 1].set_ylabel("Seconds")
        axes[0, 1].grid(True, alpha=0.3)
        
        # Final Queue
        axes[1, 0].plot(df['episode'], df['final_queue'], color='#dc2626', marker='^', markersize=3, label='Queue')
        axes[1, 0].set_title("Độ dài hàng đợi cuối episode", fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel("Episode")
        axes[1, 0].set_ylabel("Số xe")
        axes[1, 0].grid(True, alpha=0.3)
        
        # Throughput
        axes[1, 1].plot(df['episode'], df['throughput'], color='#16a34a', marker='d', markersize=3, label='Throughput')
        axes[1, 1].set_title("Lưu lượng thoát nút (Throughput)", fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel("Episode")
        axes[1, 1].set_ylabel("Số xe qua")
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        chart_path = os.path.join(latest_run, "colab_training_curves.png")
        plt.savefig(chart_path, dpi=200)
        print(f"✅ Đã lưu biểu đồ vào: {chart_path}")
        plt.show()
    else:
        print("Không tìm thấy training_log.csv trong", latest_run)


## 8. Đánh giá mô hình đã huấn luyện (`evaluate.py`)

Chạy kiểm tra chính sách học được với cờ greedy (`epsilon = 0`) trên các seed kiểm tra độc lập và xuất metrics chi tiết.

In [ ]:
%cd /content/Traffic-Light-ABM

best_weights = os.path.join(latest_run, "best_weights.pt")
if not os.path.exists(best_weights):
    best_weights = os.path.join(latest_run, "dqn_weights.pt")

if os.path.exists(best_weights):
    print("Đang đánh giá với checkpoint:", best_weights)
    !python3 dqn_py/evaluate.py \
        --weights "$best_weights" \
        --episodes 5 \
        --start-server \
        --port 6870
else:
    print("Chưa có file weights để đánh giá.")